# F2.3 on Kaggle: InceptionTime sequence baseline

Runs the repo's scripts on a Kaggle GPU. All logic lives in the repo; this notebook only calls it.

Before running:
1. **Add Input**: dataset `hooong/aviation-maintenance-dataset-from-the-ngafid`.
2. **Settings → Accelerator**: GPU. **Settings → Internet**: on.
3. Only if the GitHub repo is private: **Add-ons → Secrets**, add `GITHUB_TOKEN` (read-only access to this repo) and attach it to the notebook.

In [ ]:
!nvidia-smi -L
!find /kaggle/input -maxdepth 3 | head -40

## 1. Get the code and check the environment

In [ ]:
import subprocess

REPO = "YesimNurT/maintenance-triage-copilot"
TARGET = "/kaggle/working/repo"
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass  # public repo: no token needed

!rm -rf /kaggle/working/repo
auth = f"{token}@" if token else ""
result = subprocess.run(
    ["git", "clone", "--depth", "1", f"https://{auth}github.com/{REPO}.git", TARGET],
    capture_output=True,  # never print the URL: it may hold the token
)
assert result.returncode == 0, "clone failed: is the repo public, or is GITHUB_TOKEN attached?"
# drop the token from the saved git config
public_url = f"https://github.com/{REPO}.git"
subprocess.run(["git", "-C", TARGET, "remote", "set-url", "origin", public_url], check=True)
%cd /kaggle/working/repo
!git log --oneline -1
!pip install -q pydantic-settings python-dotenv
!pip install -q --no-deps -e .
!python -m pytest -q

## 2. Point the settings at Kaggle paths and the GPU

In [ ]:
import os

os.environ.update({
    "NGAFID_RAW_DIR": "/kaggle/working/raw",
    "DATA_DIR": "/kaggle/working/data",
    "RESULTS_DIR": "/kaggle/working/results",
    "DEVICE": "cuda",
    "SEQ_BATCH_SIZE": "64",
})
!python scripts/kaggle_prepare_raw.py /kaggle/input

## 3. Splits, per-flight features (quality filter), 1 Hz windows

In [ ]:
!python scripts/make_splits.py
!python scripts/build_features.py
!python scripts/build_sequences.py

## 4. Train: without oil first (the deciding experiment), then the other two

In [ ]:
!python scripts/f2_sequence_baseline.py --channels without_oil

In [ ]:
!python scripts/f2_sequence_baseline.py --channels all

In [ ]:
!python scripts/f2_sequence_baseline.py --channels oil_only

## 5. Summary and download

In [ ]:
import json
from pathlib import Path

for path in sorted(Path("/kaggle/working/results/f2").glob("sequence_*.json")):
    r = json.loads(path.read_text())
    low, high = r["val"]["auc_ci95"]
    val = r["val"]
    print(f"{path.stem:24s} val auc {val['auc']:.3f} [{low:.3f}, {high:.3f}]  "
          f"acc {val['accuracy']:.3f}  epochs {len(r['history'])}  "
          f"signal {r['signal_present']}")
!cd /kaggle/working && zip -qr results.zip results && ls -la results.zip